In [1]:
"""
05_hyperparameter_tuning.ipynb

Purpose:
Tune the baseline models using cross-validated hyperparameter search
on the training split, then evaluate tuned models on the validation split.

Models tuned:
1. Random Forest
2. XGBoost
3. LightGBM

What this notebook does:
1. Load train and validation splits
2. Define parameter search spaces
3. Run RandomizedSearchCV with StratifiedKFold
4. Optimize Macro F1
5. Evaluate tuned models on validation data
6. Compare tuned models fairly
7. Save only the important tuned models and summary reports

"""

# 1. Imports

# sys is used only to print Python version
import sys

# Path helps build file paths clearly
from pathlib import Path

# warnings are hidden to keep notebook output cleaner
import warnings

# json is used to save best parameters
import json

# joblib is used to save trained models
import joblib

# numpy and pandas are used for data handling
import numpy as np
import pandas as pd

# RandomizedSearchCV runs the parameter search
# StratifiedKFold keeps class balance in cross-validation folds
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

# Metrics used to compare tuned models
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)

# Used for multiclass ROC-AUC
from sklearn.preprocessing import label_binarize

# Baseline model 1
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

# display() looks nicer in notebooks, but print() is used if unavailable
try:
    from IPython.display import display
except Exception:
    display = None

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)


# 2. Optional Model Imports

# These libraries may not always be installed
HAS_XGBOOST = True
HAS_LIGHTGBM = True

try:
    from xgboost import XGBClassifier
except Exception:
    HAS_XGBOOST = False
    print("XGBoost not available. Skipping XGBClassifier.")

try:
    from lightgbm import LGBMClassifier
except Exception:
    HAS_LIGHTGBM = False
    print("LightGBM not available. Skipping LGBMClassifier.")


# 3. Paths

PROJECT_ROOT = Path.cwd().parent
SPLITS_PATH = PROJECT_ROOT / "data" / "splits"
REPORTS_PATH = PROJECT_ROOT / "reports" / "hyperparameter_tuning"
MODELS_PATH = PROJECT_ROOT / "artifacts" / "tuned_models"

REPORTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("SPLITS_PATH  :", SPLITS_PATH)
print("REPORTS_PATH :", REPORTS_PATH)
print("MODELS_PATH  :", MODELS_PATH)


# 4. Helper Functions

def show_df(df: pd.DataFrame, n: int = 5) -> None:
    """
    Show the first few rows of a dataframe.
    """
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n))


def save_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """
    Save dataframe as CSV.
    """
    df.to_csv(path, index=index)


def evaluate_multiclass(y_true, y_pred, y_proba, class_labels):
    """
    Compute the main multi-class evaluation metrics.

    Macro F1 is the main selection metric because the dataset is imbalanced
    and we want fair performance across classes.
    """
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    # Convert multiclass labels into binary form for ROC-AUC
    y_true_bin = label_binarize(y_true, classes=class_labels)

    result["roc_auc_ovr_macro"] = roc_auc_score(
        y_true_bin,
        y_proba,
        multi_class="ovr",
        average="macro",
    )

    result["roc_auc_ovr_weighted"] = roc_auc_score(
        y_true_bin,
        y_proba,
        multi_class="ovr",
        average="weighted",
    )

    return result


def get_search_spaces():
    """
    Define the hyperparameter search space for each model.

    These are the parameter ranges tested by RandomizedSearchCV.
    """
    spaces = {}

    # Random Forest search space
    spaces["random_forest"] = {
        "model": RandomForestClassifier(
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "params": {
            "n_estimators": [200, 300, 400, 500, 700],
            "max_depth": [None, 10, 20, 30, 40],
            "min_samples_split": [2, 5, 10, 15],
            "min_samples_leaf": [1, 2, 4, 6],
            "max_features": ["sqrt", "log2", None],
            "bootstrap": [True, False],
        },
    }

    # XGBoost search space
    if HAS_XGBOOST:
        spaces["xgboost"] = {
            "model": XGBClassifier(
                objective="multi:softprob",
                num_class=3,
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
            ),
            "params": {
                "n_estimators": [200, 300, 400, 500, 700],
                "max_depth": [3, 4, 5, 6, 8],
                "learning_rate": [0.01, 0.03, 0.05, 0.1],
                "subsample": [0.7, 0.8, 0.9, 1.0],
                "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
                "min_child_weight": [1, 3, 5, 7],
                "gamma": [0, 0.1, 0.3, 0.5],
                "reg_alpha": [0, 0.01, 0.1, 1.0],
                "reg_lambda": [1.0, 1.5, 2.0, 3.0],
            },
        }

    # LightGBM search space
    if HAS_LIGHTGBM:
        spaces["lightgbm"] = {
            "model": LGBMClassifier(
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbosity=-1,
            ),
            "params": {
                "n_estimators": [200, 300, 400, 500, 700],
                "learning_rate": [0.01, 0.03, 0.05, 0.1],
                "num_leaves": [15, 31, 63, 127],
                "max_depth": [-1, 5, 10, 20, 30],
                "min_child_samples": [10, 20, 30, 50, 80],
                "subsample": [0.7, 0.8, 0.9, 1.0],
                "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
                "reg_alpha": [0.0, 0.01, 0.1, 1.0],
                "reg_lambda": [0.0, 0.01, 0.1, 1.0],
            },
        }

    return spaces


# 5. Load Splits

# Load training and validation feature sets
X_train = pd.read_csv(SPLITS_PATH / "X_train.csv")
X_valid = pd.read_csv(SPLITS_PATH / "X_valid.csv")

# Load target labels
y_train = pd.read_csv(SPLITS_PATH / "y_train.csv")["fault_severity"]
y_valid = pd.read_csv(SPLITS_PATH / "y_valid.csv")["fault_severity"]

print("\nLoaded splits:")
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

# Make sure features match
if list(X_train.columns) != list(X_valid.columns):
    raise ValueError("Feature mismatch between X_train and X_valid.")

# Sorted class labels
class_labels = sorted(y_train.unique().tolist())
print("Classes:", class_labels)


# 6. Cross-Validation Strategy

# Use 5-fold stratified cross-validation
# This means each fold keeps a similar class distribution
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("\nCross-validation strategy:")
print(cv)


# 7. Hyperparameter Search

search_spaces = get_search_spaces()

if len(search_spaces) == 0:
    raise ValueError("No models available for tuning.")

search_results_rows = []
validation_rows = []
per_class_rows = []

# Number of random parameter combinations tested per model
N_ITER = 20

for model_name, spec in search_spaces.items():
    print(f"\nTUNING MODEL: {model_name}")

    # RandomizedSearchCV tries random combinations from the search space
    # and selects the best one using cross-validated Macro F1
    search = RandomizedSearchCV(
        estimator=spec["model"],
        param_distributions=spec["params"],
        n_iter=N_ITER,
        scoring="f1_macro",
        n_jobs=-1,
        cv=cv,
        verbose=1,
        random_state=RANDOM_STATE,
        refit=True,
        return_train_score=True,
    )

    # Fit search on training data only
    search.fit(X_train, y_train)

    # Best tuned model after cross-validation
    best_model = search.best_estimator_
    best_params = search.best_params_
    best_cv_score = search.best_score_

    print(f"Best CV Macro F1 for {model_name}: {best_cv_score:.6f}")
    print("Best params:", best_params)

    # Save best model
    joblib.dump(best_model, MODELS_PATH / f"{model_name}_best_model.joblib")

    # Save best parameters
    with open(MODELS_PATH / f"{model_name}_best_params.json", "w", encoding="utf-8") as f:
        json.dump(best_params, f, indent=2)

    # Save one summary row for CV search
    search_results_rows.append(
        {
            "model": model_name,
            "best_cv_f1_macro": round(best_cv_score, 6),
            "best_index": int(search.best_index_),
            "n_candidates": int(len(pd.DataFrame(search.cv_results_))),
        }
    )

    # Evaluate tuned model on validation set
    y_pred = best_model.predict(X_valid)
    y_proba = best_model.predict_proba(X_valid)

    metrics = evaluate_multiclass(
        y_true=y_valid,
        y_pred=y_pred,
        y_proba=y_proba,
        class_labels=class_labels,
    )

    row = {"model": model_name}
    row.update({k: round(v, 6) for k, v in metrics.items()})
    validation_rows.append(row)

    # Save per-class metrics
    clf_rep = classification_report(
        y_valid,
        y_pred,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )

    for cls in class_labels:
        cls_key = str(cls)
        per_class_rows.append(
            {
                "model": model_name,
                "class": cls,
                "precision": clf_rep[cls_key]["precision"],
                "recall": clf_rep[cls_key]["recall"],
                "f1_score": clf_rep[cls_key]["f1-score"],
                "support": clf_rep[cls_key]["support"],
            }
        )


# 8. Summaries

# Cross-validation summary
cv_summary_df = pd.DataFrame(search_results_rows).sort_values(
    by="best_cv_f1_macro",
    ascending=False,
).reset_index(drop=True)

# Validation summary
validation_summary_df = pd.DataFrame(validation_rows).sort_values(
    by=["f1_macro", "balanced_accuracy", "roc_auc_ovr_macro"],
    ascending=False,
).reset_index(drop=True)

# Per-class summary
per_class_df = pd.DataFrame(per_class_rows)

print("\nCROSS-VALIDATION SUMMARY")
print(cv_summary_df)

print("\nVALIDATION SUMMARY OF TUNED MODELS")
print(validation_summary_df)

save_csv(cv_summary_df, REPORTS_PATH / "tuning_cv_summary.csv", index=False)
save_csv(validation_summary_df, REPORTS_PATH / "tuned_validation_summary.csv", index=False)
save_csv(per_class_df, REPORTS_PATH / "tuned_per_class_metrics.csv", index=False)

if display is not None:
    display(cv_summary_df)
    display(validation_summary_df)
    display(per_class_df.head(12))


# 9. Best Tuned Model

# Select the best tuned model using highest validation Macro F1
best_model_name = validation_summary_df.iloc[0]["model"]

best_model_summary = pd.DataFrame(
    {
        "selection_rule": ["Highest validation Macro F1 after CV tuning"],
        "best_model": [best_model_name],
        "f1_macro": [validation_summary_df.iloc[0]["f1_macro"]],
        "balanced_accuracy": [validation_summary_df.iloc[0]["balanced_accuracy"]],
        "roc_auc_ovr_macro": [validation_summary_df.iloc[0]["roc_auc_ovr_macro"]],
    }
)

print("\nBEST TUNED MODEL")
print(best_model_summary)

save_csv(best_model_summary, REPORTS_PATH / "best_tuned_model_summary.csv", index=False)


# 10. Final Notebook Summary

# Small summary table for viva
summary_df = pd.DataFrame(
    {
        "item": [
            "n_train_rows",
            "n_valid_rows",
            "n_features",
            "n_models_tuned",
            "cv_folds",
            "search_iterations_per_model",
            "primary_metric",
            "best_tuned_model",
        ],
        "value": [
            len(X_train),
            len(X_valid),
            X_train.shape[1],
            len(search_spaces),
            5,
            N_ITER,
            "f1_macro",
            best_model_name,
        ],
    }
)

print("\nNOTEBOOK SUMMARY")
print(summary_df)

save_csv(summary_df, REPORTS_PATH / "tuning_notebook_summary.csv", index=False)

print("\nHYPERPARAMETER TUNING COMPLETE")
print("Reports saved to :", REPORTS_PATH)
print("Models saved to  :", MODELS_PATH)

Python : 3.12.2
Pandas : 2.3.3
NumPy  : 2.3.5
PROJECT_ROOT : /Users/hasheenadilmidesilva/Desktop/NetGuard
SPLITS_PATH  : /Users/hasheenadilmidesilva/Desktop/NetGuard/data/splits
REPORTS_PATH : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/hyperparameter_tuning
MODELS_PATH  : /Users/hasheenadilmidesilva/Desktop/NetGuard/artifacts/tuned_models

Loaded splits:
X_train: (4723, 860)
X_valid: (1181, 860)
y_train: (4723,)
y_valid: (1181,)
Classes: [0, 1, 2]

Cross-validation strategy:
StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

TUNING MODEL: random_forest
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV Macro F1 for random_forest: 0.697132
Best params: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 40, 'bootstrap': True}

TUNING MODEL: xgboost
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best CV Macro F1 for xgboost: 0.687546
Best params: {'subsample': 0.9, 'reg_lambda': 1.0,

,model,best_cv_f1_macro,best_index,n_candidates
0,lightgbm,0.703146,8,20
1,random_forest,0.697132,2,20
2,xgboost,0.687546,16,20


,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr_macro,roc_auc_ovr_weighted
0,lightgbm,0.744285,0.739017,0.690128,0.739017,0.707817,0.773797,0.744285,0.752550,0.882571,0.860530
1,random_forest,0.744285,0.734378,0.677128,0.734378,0.700061,0.764286,0.744285,0.750496,0.888579,0.867674
2,xgboost,0.759526,0.691320,0.706158,0.691320,0.697994,0.753637,0.759526,0.755877,0.890181,0.870357


,model,class,precision,recall,f1_score,support
0,random_forest,0,0.863636,0.768930,0.813536,766.0
1,random_forest,1,0.577143,0.675585,0.622496,299.0
2,random_forest,2,0.590604,0.758621,0.664151,116.0
3,xgboost,0,0.820000,0.856397,0.837803,766.0
4,xgboost,1,0.608209,0.545151,0.574956,299.0
5,xgboost,2,0.690265,0.672414,0.681223,116.0
6,lightgbm,0,0.880916,0.753264,0.812104,766.0
7,lightgbm,1,0.552430,0.722408,0.626087,299.0
8,lightgbm,2,0.637037,0.741379,0.685259,116.0



BEST TUNED MODEL
                                selection_rule best_model  f1_macro  balanced_accuracy  roc_auc_ovr_macro
0  Highest validation Macro F1 after CV tuning   lightgbm  0.707817           0.739017           0.882571

NOTEBOOK SUMMARY
                          item     value
0                 n_train_rows      4723
1                 n_valid_rows      1181
2                   n_features       860
3               n_models_tuned         3
4                     cv_folds         5
5  search_iterations_per_model        20
6               primary_metric  f1_macro
7             best_tuned_model  lightgbm

HYPERPARAMETER TUNING COMPLETE
Reports saved to : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/hyperparameter_tuning
Models saved to  : /Users/hasheenadilmidesilva/Desktop/NetGuard/artifacts/tuned_models
